# H-N40 — BHSD-only nnU-Net baseline
Dataset002_BHSDOnly을 H-N00과 같은 기본 nnU-Net 조건으로 학습합니다. 학습은 분리된 백그라운드 프로세스로 실행되며 이 노트북을 다시 열어도 `progress.json`으로 진행률을 확인할 수 있습니다.

In [7]:
# 1. 경로 및 설정
import json
import os
import subprocess
import time
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import clear_output, display

REPO_ROOT = Path('/home/stu03/workspace/model').resolve()
CONFIG_PATH = REPO_ROOT / 'configs/experiments/hemorrhage/H-N40.yaml'
DATA_ROOT = Path('/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet').resolve()
RESULTS_ROOT = REPO_ROOT / 'runs/nnunet'
PYTHON = REPO_ROOT / '.venv/bin/python'
LOG_PATH = RESULTS_ROOT / 'H-N40-launch.log'
PID_PATH = RESULTS_ROOT / 'H-N40.pid'

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
experiment = config['experiment']
plans = config['plans']
dataset_folder = f"Dataset{int(experiment['dataset_id']):03d}_{experiment['dataset_name']}"
trainer_folder = f"nnUNetTrainerICHBaseline__{plans['output_identifier']}__{experiment['configuration']}"
OUTPUT_FOLDER = RESULTS_ROOT / dataset_folder / trainer_folder / f"fold_{experiment['fold']}"
PROGRESS_PATH = OUTPUT_FOLDER / 'progress.json'
METRICS_PATH = OUTPUT_FOLDER / 'custom_metrics.json'
PLAN_PATH = DATA_ROOT / 'nnUNet_preprocessed' / dataset_folder / f"{plans['output_identifier']}.json"
print('Config:', CONFIG_PATH)
print('Plan:', PLAN_PATH)
print('Output:', OUTPUT_FOLDER)

Config: /home/stu03/workspace/model/configs/experiments/hemorrhage/H-N40.yaml
Plan: /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed/Dataset002_BHSDOnly/ICHPlansBHSDBaselineV1.json
Output: /home/stu03/workspace/model/runs/nnunet/Dataset002_BHSDOnly/nnUNetTrainerICHBaseline__ICHPlansBHSDBaselineV1__3d_fullres/fold_0


In [8]:
# 2. 실행 전 검사
required = [CONFIG_PATH, PYTHON, DATA_ROOT / 'nnUNet_raw' / dataset_folder, DATA_ROOT / 'nnUNet_preprocessed' / dataset_folder, PLAN_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('필수 경로가 없습니다:\n' + '\n'.join(missing))
assert experiment['dataset_id'] == 2
assert experiment['dataset_name'] == 'BHSDOnly'
assert config['network']['name'] == 'default'
assert plans['output_identifier'] == 'ICHPlansBHSDBaselineV1'
print('epochs:', config['trainer']['epochs'])
print('save_every:', config['trainer']['save_every'])
print('early stopping:', config['pruning'])
print('H-N40 실행 준비 완료')

epochs: 300
save_every: 5
early stopping: {'enabled': True, 'monitor': 'composite', 'min_epochs': 200, 'patience': 30, 'min_delta': 0.001}
H-N40 실행 준비 완료


In [9]:
# 3. 자식 프로세스 환경과 dry-run
environment = os.environ.copy()
environment['nnUNet_n_proc_DA'] = '0'
environment['nnUNet_def_n_proc'] = '1'
environment['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), environment.get('PYTHONPATH', '')]).rstrip(os.pathsep)
command = [str(PYTHON), '-m', 'brainon_ai.nnunet.runner', '--config', str(CONFIG_PATH), '--data-root', str(DATA_ROOT), '--results-root', str(RESULTS_ROOT)]
dry_run = subprocess.run(command + ['--dry-run'], cwd=REPO_ROOT, env=environment, text=True, capture_output=True)
print(dry_run.stdout)
if dry_run.returncode != 0:
    print(dry_run.stderr)
    raise RuntimeError(f'dry-run 실패: {dry_run.returncode}')

{
  "config": "/home/stu03/workspace/model/configs/experiments/hemorrhage/H-N40.yaml",
  "data_paths": {
    "nnUNet_raw": "/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_raw",
    "nnUNet_preprocessed": "/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed",
    "nnUNet_results": "/home/stu03/workspace/model/runs/nnunet"
  },
  "output_folder": "/home/stu03/workspace/model/runs/nnunet/Dataset002_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansBHSDBaselineV1__3d_fullres/fold_0",
  "experiment": {
    "name": "H-N40-nnunet-bhsd-only-baseline",
    "dataset_id": 2,
    "dataset_name": "BHSDOnly",
    "configuration": "3d_fullres",
    "fold": 0,
    "device": "auto",
    "continue_training": false,
    "validation_checkpoint": "best_composite"
  },
  "plans": {
    "base_identifier": "nnUNetPlans",
    "output_identifier": "ICHPlansBHSDBaselineV1",
    "patch_size": [
      20,
      320,
      256
    ],
    "batch_size": 1
  },
  "trai

In [10]:
# 4. 새 실행 또는 재개 가능 여부 확인
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
continue_training = bool(config['experiment'].get('continue_training', False))
latest = OUTPUT_FOLDER / 'checkpoint_latest.pth'
if OUTPUT_FOLDER.exists():
    print('기존 파일:', sorted(path.name for path in OUTPUT_FOLDER.iterdir()))
    if not continue_training:
        raise RuntimeError('기존 결과가 있습니다. 새 실행이면 폴더를 삭제하고, 재개면 YAML의 continue_training을 true로 바꾸세요.')
    if not latest.is_file():
        raise RuntimeError('재개용 checkpoint_latest.pth가 없습니다.')
    print('checkpoint_latest에서 재개합니다:', latest)
elif continue_training:
    raise RuntimeError('결과 폴더가 없으므로 continue_training은 false여야 합니다.')
else:
    print('새 H-N40 학습을 시작할 수 있습니다.')

새 H-N40 학습을 시작할 수 있습니다.


In [11]:
# 5. 백그라운드 학습 시작 — 한 번만 실행
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
log_handle = LOG_PATH.open('a', encoding='utf-8', buffering=1)
log_handle.write('\n\n===== H-N40 launch =====\n' + ' '.join(command) + '\n')
log_handle.flush()
process = subprocess.Popen(command, cwd=REPO_ROOT, env=environment, stdout=log_handle, stderr=subprocess.STDOUT, text=True, start_new_session=True)
PID_PATH.write_text(str(process.pid), encoding='utf-8')
print('H-N40 학습 시작')
print('PID:', process.pid)
print('Log:', LOG_PATH)
print('Progress:', PROGRESS_PATH)

H-N40 학습 시작
PID: 920472
Log: /home/stu03/workspace/model/runs/nnunet/H-N40-launch.log
Progress: /home/stu03/workspace/model/runs/nnunet/Dataset002_BHSDOnly/nnUNetTrainerICHBaseline__ICHPlansBHSDBaselineV1__3d_fullres/fold_0/progress.json


In [12]:
# 6. 시작 직후 오류 확인
time.sleep(3)
return_code = process.poll()
if return_code is None:
    print('학습 프로세스 실행 중')
else:
    print('학습 프로세스 조기 종료:', return_code)
    if LOG_PATH.exists():
        print('\n'.join(LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()[-100:]))

학습 프로세스 실행 중


In [13]:
# 7. 실시간 진행률 — 커널을 다시 시작한 뒤에도 이 셀만 실행 가능
def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def format_time(seconds):
    if seconds is None:
        return '-'
    hours, remainder = divmod(max(0, int(seconds)), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

while True:
    state = read_json(PROGRESS_PATH)
    clear_output(wait=True)
    display(pd.DataFrame([{
        '실험': 'H-N40',
        '상태': state.get('status', 'starting'),
        'Epoch': f"{state.get('current_epoch', 0)}/{state.get('total_epochs', config['trainer']['epochs'])}",
        '진행률(%)': state.get('progress_percent', 0),
        'ETA': format_time(state.get('eta_seconds')),
        'Dice': state.get('pseudo_dice'),
        'Recall': state.get('pseudo_recall'),
        'Composite': state.get('composite'),
        'Best Dice': state.get('best_dice'),
        'Best Recall': state.get('best_recall'),
        'Best Composite': state.get('best_composite'),
        'Early Stop': f"{state.get('pruning_bad_checks', 0)}/{state.get('pruning_patience', 30)}",
        'Latest 저장': (OUTPUT_FOLDER / 'checkpoint_latest.pth').is_file(),
    }]))
    if state.get('message'):
        print('message:', state['message'])
    if state.get('status') in {'completed', 'early_stopped', 'failed'}:
        break
    if 'process' in globals() and process.poll() is not None:
        print('process return code:', process.poll())
        break
    time.sleep(10)

,실험,상태,Epoch,진행률(%),ETA,Dice,Recall,Composite,Best Dice,Best Recall,Best Composite,Early Stop,Latest 저장
0,H-N40,early_stopped,237/300,79.0,00:19:09,None,None,None,0.830074,0.848239,0.839156,30/30,False


In [ ]:
# 8. 로그 마지막 100줄
if LOG_PATH.exists():
    print('\n'.join(LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()[-100:]))
else:
    print('로그 파일이 아직 없습니다.')

In [ ]:
# 9. 실제 full-volume Dice와 기존 H-N00의 동일 BHSD 28건 비교
BASELINE_SUMMARY = REPO_ROOT / 'runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansBaselineV1__3d_fullres/fold_0/validation/summary.json'
H_N40_SUMMARY = OUTPUT_FOLDER / 'validation/summary.json'

def positive_macro_dice(summary_path, prefix='BHSD_'):
    summary = read_json(summary_path)
    values = []
    for case in summary.get('metric_per_case', []):
        reference = case.get('reference_file', '').replace('\\', '/').rsplit('/', 1)[-1]
        metric = case['metrics']['1']
        if reference.startswith(prefix) and metric.get('n_ref', 0) > 0:
            values.append(float(metric['Dice']))
    return (sum(values) / len(values), len(values)) if values else (None, 0)

baseline_dice, baseline_n = positive_macro_dice(BASELINE_SUMMARY)
h_n40_dice, h_n40_n = positive_macro_dice(H_N40_SUMMARY)
comparison = pd.DataFrame([
    {'실험': 'H-N00 mixed-data baseline', 'BHSD 양성 건수': baseline_n, 'BHSD Macro Dice': baseline_dice},
    {'실험': 'H-N40 BHSD-only baseline', 'BHSD 양성 건수': h_n40_n, 'BHSD Macro Dice': h_n40_dice},
])
display(comparison)
if baseline_dice is not None and h_n40_dice is not None:
    print('H-N40 - H-N00:', round(h_n40_dice - baseline_dice, 6))